# Curated Taxi Data Exploration on `trips_clean`

This notebook explores a manageable partition-pruned slice from the curated `trips_clean` cache. The cache still represents the full 2021-2025 normalized dataset, but the notebook does not load the entire cache into one interactive session by default.


## 1. Imports and Runtime Parameters

In [1]:
import sys
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import pandas as pd

WORK_ROOT = Path.cwd().resolve()
if not (WORK_ROOT / "src").exists():
    WORK_ROOT = WORK_ROOT.parent
SRC_DIR = WORK_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from trajectory_utils import load_trajectory_df, register_trajectory_view

SPARK_MASTER = "spark://spark-master:7077"
MINIO_ENDPOINT = "http://minio1:9000"
MINIO_ACCESS_KEY = "minioadmin"
MINIO_SECRET_KEY = "minioadmin"
BUCKET = "taxi-data"
RAW_SOURCE_PREFIXES = ["2021", "2022", "2023", "2024", "2025"]
# NORMALIZED_CACHE_PREFIX: full curated cache cho toan bo 2021-2025 neu da duoc build san.
# EXPLORATION_CACHE_BASE_PREFIX: cache nho hon de notebook tu su dung cho pham vi dang EDA.
# REFRESH_NORMALIZED_CACHE=True: xoa cache scope hien tai va build lai tu raw data da chon.
NORMALIZED_CACHE_PREFIX = "notebook_cache/trips_clean_2021_2025"
EXPLORATION_CACHE_BASE_PREFIX = "notebook_cache/exploration_trips_clean"
USE_NORMALIZED_CACHE = True
REFRESH_NORMALIZED_CACHE = False
ALLOW_RAW_BUILD_IN_NOTEBOOK = True
RUN_FULL_ROW_COUNT = False
SAMPLE_FRACTION = 0.001
SAMPLE_SEED = 42
# Notebook chi doc mot lat cat tu curated data de phu hop voi may RAM 16 GB.
EXPLORE_TRIP_YEARS = [2024]
EXPLORE_TRIP_MONTHS = [1, 2, 3]


## 2. Create Spark Session

In [2]:
try:
    spark.stop()
except Exception:
    pass

spark = (
    SparkSession.builder.master(SPARK_MASTER).appName("notebook-full-taxi-exploration")
    .config("spark.hadoop.fs.s3a.endpoint", MINIO_ENDPOINT)
    .config("spark.hadoop.fs.s3a.access.key", MINIO_ACCESS_KEY)
    .config("spark.hadoop.fs.s3a.secret.key", MINIO_SECRET_KEY)
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")
spark


:: loading settings :: url = jar:file:/usr/local/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/jovyan/.ivy2/cache
The jars for the packages stored in: /home/jovyan/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
software.amazon.awssdk#bundle added as a dependency
org.apache.spark#spark-avro_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-7fcfc532-d059-4f52-aa89-b44315ccc178;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.4.1 in central
	found software.amazon.awssdk#bundle;2.24.6 in central
	found org.wildfly.openssl#wildfly-openssl;1.1.3.Final in central
	found org.apache.spark#spark-avro_2.13;4.0.1 in central
	found org.scala-lang.modules#scala-parallel-collections_2.13;1.2.0 in central
	found org.tukaani#xz;1.10 in central
:: resolution report :: resolve 350ms :: artifacts dl 5ms
	:: modules in use:
	org.apache.hadoop#hadoop-aw

## 3. Load a Partition-Pruned Exploration Slice from the Curated Cache


In [3]:
def s3_path_exists(spark_session, path_str):
    path = spark_session._jvm.org.apache.hadoop.fs.Path(path_str)
    filesystem = path.getFileSystem(spark_session._jsc.hadoopConfiguration())
    return filesystem.exists(path)


def delete_s3_path(spark_session, path_str):
    path = spark_session._jvm.org.apache.hadoop.fs.Path(path_str)
    filesystem = path.getFileSystem(spark_session._jsc.hadoopConfiguration())
    if filesystem.exists(path):
        filesystem.delete(path, True)


def format_scope(values):
    if not values:
        return "all"
    return ", ".join(str(value) for value in values)


def apply_exploration_scope(df):
    scoped_df = df
    if EXPLORE_TRIP_YEARS:
        scoped_df = scoped_df.where(F.col("trip_year").isin(EXPLORE_TRIP_YEARS))
    if EXPLORE_TRIP_MONTHS:
        scoped_df = scoped_df.where(F.col("trip_month").isin(EXPLORE_TRIP_MONTHS))
    return scoped_df


def build_scope_tag(years, months):
    years_part = "all-years" if not years else "years-" + "-".join(str(year) for year in years)
    months_part = "all-months" if not months else "months-" + "-".join(f"{month:02d}" for month in months)
    return f"{years_part}_{months_part}"


def selected_raw_prefixes():
    if EXPLORE_TRIP_YEARS and EXPLORE_TRIP_MONTHS:
        return [f"{year}/{month:02d}" for year in EXPLORE_TRIP_YEARS for month in EXPLORE_TRIP_MONTHS]
    if EXPLORE_TRIP_YEARS:
        return [str(year) for year in EXPLORE_TRIP_YEARS]
    return RAW_SOURCE_PREFIXES


full_cache_path = f"s3a://{BUCKET}/{NORMALIZED_CACHE_PREFIX}"
scope_tag = build_scope_tag(EXPLORE_TRIP_YEARS, EXPLORE_TRIP_MONTHS)
slice_cache_prefix = f"{EXPLORATION_CACHE_BASE_PREFIX}/{scope_tag}"
slice_cache_path = f"s3a://{BUCKET}/{slice_cache_prefix}"
full_cache_exists = s3_path_exists(spark, full_cache_path)
slice_cache_exists = s3_path_exists(spark, slice_cache_path)
cache_path = full_cache_path if full_cache_exists and not REFRESH_NORMALIZED_CACHE else slice_cache_path
build_cache_command = "bash /home/vanh/data/projects/bigdata/src/build_notebook_cache.sh"
cache_action = "read_full_cache_slice"
data_source_mode = "full_curated_cache"
missing_raw_prefixes = []
source_paths = []
exploration_scope = f"years={format_scope(EXPLORE_TRIP_YEARS)} | months={format_scope(EXPLORE_TRIP_MONTHS)}"

if USE_NORMALIZED_CACHE and full_cache_exists and not REFRESH_NORMALIZED_CACHE:
    print("Cache status: HIT (full curated cache)")
    print("Loading curated cache from:", full_cache_path)
    print("Exploration scope:", exploration_scope)
    trajectory_df = apply_exploration_scope(spark.read.parquet(full_cache_path))
elif USE_NORMALIZED_CACHE and slice_cache_exists and not REFRESH_NORMALIZED_CACHE:
    cache_action = "read_slice_cache"
    data_source_mode = "slice_cache"
    cache_path = slice_cache_path
    print("Cache status: HIT (exploration slice cache)")
    print("Loading exploration cache from:", slice_cache_path)
    print("Exploration scope:", exploration_scope)
    trajectory_df = spark.read.parquet(slice_cache_path)
else:
    if not ALLOW_RAW_BUILD_IN_NOTEBOOK:
        cache_action = "refresh_and_rebuild" if REFRESH_NORMALIZED_CACHE else "cache_missing"
        data_source_mode = "cache_required"
        print("Cache status:", "REFRESH" if REFRESH_NORMALIZED_CACHE else "MISS")
        print("No suitable cache exists for the selected exploration scope.")
        print("You can either:")
        print("- Build the full curated cache first with:")
        print(build_cache_command)
        print("- Or set ALLOW_RAW_BUILD_IN_NOTEBOOK=True to let the notebook build only the selected scope.")
        raise RuntimeError("A cache is required or ALLOW_RAW_BUILD_IN_NOTEBOOK must be True.")

    cache_action = "refresh_slice_cache" if REFRESH_NORMALIZED_CACHE else "build_slice_cache"
    data_source_mode = "raw_scope_then_slice_cache"
    cache_path = slice_cache_path
    candidate_prefixes = selected_raw_prefixes()
    for prefix in candidate_prefixes:
        path = f"s3a://{BUCKET}/{prefix}"
        if s3_path_exists(spark, path):
            source_paths.append(path)
        else:
            missing_raw_prefixes.append(prefix)

    if not source_paths:
        raise RuntimeError("No raw source prefixes were found for the selected exploration scope.")

    print("Cache status:", "REFRESH" if REFRESH_NORMALIZED_CACHE else "MISS")
    print("Building a small exploration cache for:", exploration_scope)
    print("Selected raw prefixes:", ", ".join(path.replace(f"s3a://{BUCKET}/", "") for path in source_paths))
    if missing_raw_prefixes:
        print("Missing raw prefixes skipped:", ", ".join(missing_raw_prefixes))

    normalized_dfs = []
    for path in source_paths:
        print("Normalizing raw trajectory data from:", path)
        normalized_dfs.append(
            load_trajectory_df(spark, "parquet", path, layout="raw", merge_schema=True)
        )

    trajectory_df = normalized_dfs[0]
    for yearly_df in normalized_dfs[1:]:
        trajectory_df = trajectory_df.unionByName(yearly_df, allowMissingColumns=True)

    trajectory_df = apply_exploration_scope(trajectory_df).repartition("trip_year", "trip_month")
    if REFRESH_NORMALIZED_CACHE:
        delete_s3_path(spark, slice_cache_path)
    print("Writing exploration cache to:", slice_cache_path)
    (
        trajectory_df.write.mode("overwrite")
        .partitionBy("trip_year", "trip_month")
        .parquet(slice_cache_path)
    )
    print("Exploration cache write completed.")
    trajectory_df = spark.read.parquet(slice_cache_path)


cache_exists_before_load = full_cache_exists or slice_cache_exists
cache_source = "full_curated_cache" if data_source_mode == "full_curated_cache" else "exploration_slice_cache"
df_clean = register_trajectory_view(spark, trajectory_df)
df_clean


SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


Cache status: MISS
Building a small exploration cache for: years=2024 | months=1, 2, 3
Selected raw prefixes: 2024/01, 2024/02, 2024/03
Normalizing raw trajectory data from: s3a://taxi-data/2024/01


Normalizing raw trajectory data from: s3a://taxi-data/2024/02
Normalizing raw trajectory data from: s3a://taxi-data/2024/03
Writing exploration cache to: s3a://taxi-data/notebook_cache/exploration_trips_clean/years-2024_months-01-02-03


Exploration cache write completed.


DataFrame[vendor_name_norm: string, payment_type_norm: string, pickup_ts: timestamp, dropoff_ts: timestamp, trip_date: date, origin_zone_id: int, destination_zone_id: int, passenger_count: bigint, trip_distance: double, fare_amt: double, surcharge: double, tip_amt: double, tolls_amt: double, total_amt: double, start_lon: double, start_lat: double, end_lon: double, end_lat: double, has_gps_coordinates: boolean, trajectory_mode: string, route_key: string, schema_family: string, trip_year: int, trip_month: int, origin_zone_bucket: int]

In [4]:
cache_status_df = pd.DataFrame(
    [
        {
            "cache_enabled": USE_NORMALIZED_CACHE,
            "full_cache_path": full_cache_path,
            "slice_cache_path": slice_cache_path,
            "full_cache_exists": full_cache_exists,
            "slice_cache_exists": slice_cache_exists,
            "cache_exists_before_load": cache_exists_before_load,
            "cache_action": cache_action,
            "cache_source": cache_source,
            "explore_trip_years": format_scope(EXPLORE_TRIP_YEARS),
            "explore_trip_months": format_scope(EXPLORE_TRIP_MONTHS),
            "selected_raw_prefixes": ", ".join(path.replace(f"s3a://{BUCKET}/", "") for path in source_paths) if source_paths else "n/a",
            "missing_raw_prefixes": ", ".join(missing_raw_prefixes) if missing_raw_prefixes else "none",
            "data_source_mode": data_source_mode,
            "allow_raw_build_in_notebook": ALLOW_RAW_BUILD_IN_NOTEBOOK,
        }
    ]
)
display(cache_status_df)


,cache_enabled,full_cache_path,slice_cache_path,full_cache_exists,slice_cache_exists,cache_exists_before_load,cache_action,cache_source,explore_trip_years,explore_trip_months,selected_raw_prefixes,missing_raw_prefixes,data_source_mode,allow_raw_build_in_notebook
0,True,s3a://taxi-data/notebook_cache/trips_clean_202...,s3a://taxi-data/notebook_cache/exploration_tri...,False,False,False,build_slice_cache,exploration_slice_cache,2024,"1, 2, 3","2024/01, 2024/02, 2024/03",none,raw_scope_then_slice_cache,True


## 4. Quick Structural Inspection

In [5]:
df_clean.printSchema()
print("Column count:", len(df_clean.columns))
if RUN_FULL_ROW_COUNT:
    print("Row count:", df_clean.count())
else:
    print("Row count: skipped by default on the full 2021-2025 dataset (set RUN_FULL_ROW_COUNT = True to enable full count)")


root
 |-- vendor_name_norm: string (nullable = true)
 |-- payment_type_norm: string (nullable = true)
 |-- pickup_ts: timestamp (nullable = true)
 |-- dropoff_ts: timestamp (nullable = true)
 |-- trip_date: date (nullable = true)
 |-- origin_zone_id: integer (nullable = true)
 |-- destination_zone_id: integer (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- fare_amt: double (nullable = true)
 |-- surcharge: double (nullable = true)
 |-- tip_amt: double (nullable = true)
 |-- tolls_amt: double (nullable = true)
 |-- total_amt: double (nullable = true)
 |-- start_lon: double (nullable = true)
 |-- start_lat: double (nullable = true)
 |-- end_lon: double (nullable = true)
 |-- end_lat: double (nullable = true)
 |-- has_gps_coordinates: boolean (nullable = true)
 |-- trajectory_mode: string (nullable = true)
 |-- route_key: string (nullable = true)
 |-- schema_family: string (nullable = true)
 |-- trip_year: integer (nullable 

In [6]:
df_clean.select(
    "vendor_name_norm",
    "payment_type_norm",
    "pickup_ts",
    "trip_date",
    "trip_year",
    "trip_month",
    "origin_zone_id",
    "destination_zone_id",
    "trip_distance",
    "total_amt",
    "route_key",
).show(10, truncate=False)


[Stage 8:=============================>                             (2 + 2) / 4]

+----------------+-----------------+-------------------+----------+---------+----------+--------------+-------------------+-------------+---------+---------+
|vendor_name_norm|payment_type_norm|pickup_ts          |trip_date |trip_year|trip_month|origin_zone_id|destination_zone_id|trip_distance|total_amt|route_key|
+----------------+-----------------+-------------------+----------+---------+----------+--------------+-------------------+-------------+---------+---------+
|VTS             |CREDIT           |2024-02-01 00:00:39|2024-02-01|2024     |2         |186           |79                 |2.22         |22.2     |186->79  |
|VTS             |CREDIT           |2024-02-29 20:58:59|2024-02-29|2024     |2         |237           |239                |1.77         |17.2     |237->239 |
|VTS             |CASH             |2024-02-01 00:00:17|2024-02-01|2024     |2         |138           |152                |8.93         |52.79    |138->152 |
|CMT             |CREDIT           |2024-02-29 20:02

## 5. Data Quality Snapshot on a Sampled View

In [7]:
sample_df = df_clean.sample(False, SAMPLE_FRACTION, seed=SAMPLE_SEED).limit(100000)
sample_size = sample_df.count()
quality_columns = [
    "pickup_ts",
    "origin_zone_id",
    "destination_zone_id",
    "trip_distance",
    "total_amt",
    "route_key",
    "trajectory_mode",
]

quality_rows = []
for column_name in quality_columns:
    null_count = sample_df.filter(F.col(column_name).isNull()).count()
    quality_rows.append((column_name, null_count, round(null_count / sample_size, 4)))

spark.createDataFrame(quality_rows, ["column_name", "null_count", "null_ratio"]).orderBy(F.desc("null_ratio")).show(truncate=False)


[Stage 33:>                                                         (0 + 8) / 8]

+-------------------+----------+----------+
|column_name        |null_count|null_ratio|
+-------------------+----------+----------+
|route_key          |0         |0.0       |
|destination_zone_id|0         |0.0       |
|total_amt          |0         |0.0       |
|pickup_ts          |0         |0.0       |
|trajectory_mode    |0         |0.0       |
|origin_zone_id     |0         |0.0       |
|trip_distance      |0         |0.0       |
+-------------------+----------+----------+



## 6. Business Summaries on the Normalized Schema

In [8]:
df_clean.groupBy("vendor_name_norm").count().orderBy(F.desc("count")).show(truncate=False)


+----------------+-------+
|vendor_name_norm|count  |
+----------------+-------+
|VTS             |7219811|
|CMT             |2334226|
|6               |720    |
+----------------+-------+



In [9]:
df_clean.groupBy("payment_type_norm").count().orderBy(F.desc("count")).show(truncate=False)


+-----------------+-------+
|payment_type_norm|count  |
+-----------------+-------+
|CREDIT           |7258145|
|CASH             |1330099|
|0                |751962 |
|DISPUTE          |152587 |
|NO CHARGE        |61964  |
+-----------------+-------+



In [10]:
df_clean.groupBy("trip_year", "trip_month").count().orderBy("trip_year", "trip_month").show(100, truncate=False)


+---------+----------+-------+
|trip_year|trip_month|count  |
+---------+----------+-------+
|2024     |1         |2964617|
|2024     |2         |3007533|
|2024     |3         |3582607|
+---------+----------+-------+



In [11]:
df_clean.select(
    F.min("trip_distance").alias("min_trip_distance"),
    F.max("trip_distance").alias("max_trip_distance"),
    F.avg("trip_distance").alias("avg_trip_distance"),
    F.min("total_amt").alias("min_total_amt"),
    F.max("total_amt").alias("max_total_amt"),
    F.avg("total_amt").alias("avg_total_amt"),
).show(truncate=False)


+-----------------+-----------------+-----------------+-------------+-------------+------------------+
|min_trip_distance|max_trip_distance|avg_trip_distance|min_total_amt|max_total_amt|avg_total_amt     |
+-----------------+-----------------+-----------------+-------------+-------------+------------------+
|0.0              |312722.3         |4.042287313009966|-1000.0      |9792.0       |26.865400463829868|
+-----------------+-----------------+-----------------+-------------+-------------+------------------+



## 7. Time-Based Exploration on the Selected Curated Slice


In [12]:
daily_summary = (
    df_clean.groupBy("trip_date")
    .agg(
        F.count("*").alias("trip_count"),
        F.avg("trip_distance").alias("avg_trip_distance"),
        F.avg("total_amt").alias("avg_total_amt"),
    )
    .orderBy("trip_date")
)
daily_summary.show(20, truncate=False)


+----------+----------+------------------+------------------+
|trip_date |trip_count|avg_trip_distance |avg_total_amt     |
+----------+----------+------------------+------------------+
|2024-01-01|81013     |4.396813968128586 |30.15371915618292 |
|2024-01-02|75519     |4.119030575087062 |30.220156781735277|
|2024-01-03|82427     |3.8784004027806858|28.60213861962545 |
|2024-01-04|102901    |3.3109689896113825|27.21558643744898 |
|2024-01-05|103178    |3.7535462986295585|26.446262963034346|
|2024-01-06|97117     |3.1258273010904576|25.085951172297065|
|2024-01-07|67543     |3.8838865611536515|28.10212990243078 |
|2024-01-08|80034     |3.51130644476093  |27.693820501286154|
|2024-01-09|93962     |3.832470253932452 |25.15785626104224 |
|2024-01-10|95000     |3.3508643157894773|26.8465066315789  |
|2024-01-11|105010    |3.5884354823350058|27.667062089322716|
|2024-01-12|103655    |4.230263663113223 |27.619334619650974|
|2024-01-13|104758    |3.7702072395425468|25.295824089806395|
|2024-01

## 8. Zone and Route Exploration on the Selected Curated Slice


In [13]:
df_clean.filter(F.col("origin_zone_id").isNotNull()).groupBy("origin_zone_id").count().orderBy(F.desc("count")).show(20, truncate=False)


+--------------+------+
|origin_zone_id|count |
+--------------+------+
|161           |453825|
|237           |439138|
|132           |429745|
|236           |416508|
|162           |336460|
|230           |331570|
|186           |319705|
|142           |316314|
|138           |284359|
|239           |281730|
|163           |276217|
|170           |272534|
|234           |257914|
|68            |253137|
|48            |246984|
|79            |234954|
|141           |229080|
|249           |228830|
|164           |215859|
|140           |199119|
+--------------+------+
only showing top 20 rows


In [14]:
df_clean.filter(F.col("route_key").isNotNull()).groupBy("route_key").count().orderBy(F.desc("count")).show(20, truncate=False)


+---------+-----+
|route_key|count|
+---------+-----+
|237->236 |64384|
|236->237 |57111|
|236->236 |47509|
|237->237 |44270|
|161->237 |31118|
|142->239 |27085|
|237->161 |26836|
|239->142 |26828|
|161->236 |26504|
|239->238 |25824|
|237->162 |24058|
|141->236 |23182|
|264->264 |21940|
|236->161 |21554|
|238->239 |21248|
|186->230 |21139|
|132->132 |21059|
|236->141 |20861|
|142->238 |20320|
|236->239 |20238|
+---------+-----+
only showing top 20 rows


## 9. Candidate Features for the Project

In [15]:
candidate_columns = [
    "trip_year",
    "trip_month",
    "origin_zone_id",
    "destination_zone_id",
    "payment_type_norm",
    "route_key",
    "origin_zone_bucket",
]

for column_name in candidate_columns:
    print(column_name)


trip_year
trip_month
origin_zone_id
destination_zone_id
payment_type_norm
route_key
origin_zone_bucket
